# AVBD 3D — Stage 11 CPU SoA/flat-grid comparisons + optional CUDA probe

Clones the **latest GitHub `main`**, builds the actual C++17 physics engine, runs all regression suites, compares the **original unordered-set isolation certificate**, the **new reusable flat occupancy grid**, and the **opt-in contiguous SoA motion predictor**. It separately compares AoS vs SoA in a **contact-rich, 324-body full AVBD solver**.

**Not a million-contact GPU test.** One million sparse engine bodies have zero contacts. The optional CUDA experiment at the end is Stage 10's translation-only kernel, not a GPU AVBD engine. No GitHub Actions or paid APIs are needed.

In Kaggle Settings, enable **Internet**. For the optional CUDA kernel, enable an **NVIDIA GPU accelerator**. The CPU experiments work without a GPU.


In [ ]:
import pathlib, subprocess, shutil, sys, csv, io, json, statistics

def command(*args, cwd=None):
    print('\n$', ' '.join(map(str, args)), flush=True)
    return subprocess.run(list(map(str,args)), cwd=cwd, check=True, text=True, capture_output=False)

repo = pathlib.Path('/kaggle/working/avbd-demo3d')
if repo.exists():
    shutil.rmtree(repo)  # Avoid stale code when re-running
command('git','clone','--depth','1','--branch','main','https://github.com/alxspiker/avbd-demo3d.git',repo)
command('git','rev-parse','HEAD',cwd=repo)
command('cmake','-S',repo,'-B',repo/'build','-DCMAKE_BUILD_TYPE=Release','-DAVBD_ENABLE_OPENMP=ON')
command('cmake','--build',repo/'build','--parallel','4')
command('ctest','--test-dir',repo/'build','--output-on-failure')


## One million *isolated* rigid bodies — full engine step

Modes: `certified` uses the Stage 10 hash-set certificate and AoS predictor. `flat` uses the flat reusable occupancy grid with the AoS predictor. `soa_flat` combines flat grid and packed SoA predictor. **The entire `World::step` runs**, including the collision separation certificate, quaternion rotation, velocity reconstruction and state validation. The certified path must report **0 contacts** and `last_certified=1`. These are **sparse**, not dense scenes. All three modes must agree on final positions.

Run **three independent passes** with four measured steps each. Time the actual code; there are no predetermined expected performance numbers.


In [ ]:
bench = repo/'build'/'avbd3d_million_benchmark'
rows=[]
for n in [10_000,100_000,1_000_000]:
    for repeat in range(3):
        modes = ['certified','flat','soa_flat'] if repeat%2==0 else ['soa_flat','flat','certified']
        for mode in modes:
            result = subprocess.run([str(bench),str(n),'4',mode],cwd=repo,text=True,capture_output=True,check=True)
            record=list(csv.DictReader(io.StringIO(result.stdout)))[0]
            assert int(record['contacts'])==0 and int(record['last_certified'])==1,record
            record['repeat']=repeat+1
            rows.append(record)
            print(record,flush=True)
for n in [10_000,100_000,1_000_000]:
    print('\nBody count:',n)
    for mode in ['certified','flat','soa_flat']:
        times=[float(r['mean_ms_step']) for r in rows if int(r['bodies'])==n and r['mode']==mode]
        print(mode,'mean ms/step:',round(statistics.mean(times),3),'runs:',[round(t,3) for t in times])
    final_y={r['last_y'] for r in rows if int(r['bodies'])==n}
    assert len(final_y)==1,'Motion differs across tested engine modes'
pathlib.Path('/kaggle/working/stage11_cpu_sparse_results.json').write_text(json.dumps(rows,indent=2))


## Full contact-rich solver — 324 bodies

Tests **contact manifolds, graph/island solving, rotations, and restitution**, not just translations. Compares the *same* scenario with the packed predictor off/on. Check `last_y`, `max_contacts`, and the trajectory-parity regression test above. The small performance difference may change with CPU hardware.


In [ ]:
dense = repo/'build'/'avbd3d_layout_benchmark'
dense_rows=[]
for repeat in range(3):
    for mode in (['aos','soa'] if repeat%2==0 else ['soa','aos']):
        result=subprocess.run([str(dense),mode,'120'],cwd=repo,text=True,capture_output=True,check=True)
        row=list(csv.DictReader(io.StringIO(result.stdout)))[0]
        print(row,flush=True)
        dense_rows.append(row)
assert len({r['last_y'] for r in dense_rows})==1,'Final dense scene positions differ'
assert len({r['max_contacts'] for r in dense_rows})==1,'Dense scene contact counts differ'
for mode in ['aos','soa']:
    times=[float(r['mean_ms_step']) for r in dense_rows if r['mode']==mode]
    print(mode,'mean full-step ms:',statistics.mean(times))
pathlib.Path('/kaggle/working/stage11_cpu_dense_results.json').write_text(json.dumps(dense_rows,indent=2))


## Optional GPU — Stage 10 translation-only kernel

This uses a Tesla T4/P100/etc if Kaggle supplies one, through CuPy/NVRTC, and is **not** an AVBD collision solver. Timing excludes PCIe transfer and GPU kernel uses `float32`, while the CPU engine uses `double`. If CuPy is missing, run `pip install -q cupy-cuda12x` **only if the Kaggle CUDA runtime supports it**, then rerun this cell. Skipping is reported as `SKIPPED`, not a fictitious GPU pass.


In [ ]:
try:
    import cupy as cp
    device_count=cp.cuda.runtime.getDeviceCount()
    print('CUDA devices:',device_count)
    if device_count:
        command(sys.executable,repo/'tools'/'gpu_freeflight_probe.py','--bodies','1000000','--steps','100',cwd=repo)
    else:
        print('GPU experiment SKIPPED: no visible CUDA devices')
except ImportError as exc:
    print('GPU experiment SKIPPED: CuPy is not installed:',exc)
except Exception as exc:
    print('GPU experiment FAILED (not counted as success):',repr(exc))
    raise


## Share results

Download `stage11_cpu_sparse_results.json`, `stage11_cpu_dense_results.json`, plus the notebook log, and upload them to ChatGPT. Include the Git commit, Kaggle CPU/GPU information and failed/skipped status rather than just the fastest timing.
